# Xarray's Rasterio backend

In this lesson, we will learn how to use xarray's rasterio backend engine to open GeoTIFF rasters. By the end of the lesson, we will be able to:

:::{admonition} Learning Goals
- Learn about the GeoTIFF format
- Lean about xarray's "rasterio" backend and the "rioxarray" accessor
- Learn how to read and plot GeoTIFF files with xarray
- Explore how to perform reprojection operations on rasters
:::

## What are GeoTIFFs?

The TIFF (Tagged Image File Format) format is a metadata rich image format for raster data. GeoTIFF (Geographic Tagged Image File Format) files are TIFF files that use georeferencing information (such as map projection and coordinate systems) as metadata. A GeoTIFF file with a single band contains 2D raster data for a single characteristic (i.e., variable) and maps to a geographic region. A GeoTIFF file can have multiple bands that all map to the same geographic region.

![Raster_Image](https://docs.qgis.org/3.44/en/_images/raster_dataset.png)

## Rasterio and Rioxarray Backends

[Rasterio](https://rasterio.readthedocs.io/en/stable/intro.html) is a geospatial raster library that expresses GDAL ([Geospatial Data Abstraction Library](http://gdal.org)) data model with a Python API and CLI. [Rioxarray](https://corteva.github.io/rioxarray/stable/readme.html) is a wrapper around the rasterio library, that also extends the xarray api with the *rio* accessor. When you open a GeoTIFF file with the "rasterio" engine it returns the data as an xarray object with access to methods from the *rio* accessor.



## Reading a GeoTIFF

Xarray's "rasterio" backend supports reading GeoTIFFs.

Lets read a GeoTIFF file as an `xr.Dataset` by selecting `engine='rasterio'`

In [ ]:
import xarray as xr

In [ ]:
rds = xr.tutorial.open_dataset("RGB.byte.tif", engine="rasterio")

:::{note} We can also read GeoTIFFs with `rioxarray.open_rasterio`.

The following code snippet opens a GeoTIFF and returns a `DataArray` object
```python
import rioxarray
rioxarray.open_rasterio("RGB.byte.tif")
```

:::

## Raster bands

GeoTIFFS can have multiple bands each representing a range or band in the electromagnetic spectrum. Arrays stored within bands in xarray are data variables and `DataArray`'s objects in the the Xarray Data Model.

Lets get the `DataArray` object (or variable) from our dataset. The variable name is "band_data".

In [ ]:
rda = rds["band_data"]
rda

Lets try getting the total number of bands for this GeoTIFF. Since `rioxarray` extends xarray with the *rio* accessor we can this accessor be able use many of the builtin `rasterio` methods.

In [ ]:
rda.rio.count

### Selection by bands
We can also select by bands. Since there are 3 bands in this dataset we can return raster array for the first band of this `xr.DataArray` by with indexing

Lets get the first band of this dataset and try plotting it

In [ ]:
rda[0].plot(cmap="pink")

## Bounds
With `.rio.bounds()` we can get the spatial bounding box of our `DataArray`

In [ ]:
rda.rio.bounds()

## Transformation

With `.rio.transform()` we can get the affine transformation matrix that maps pixel locations in (col, row) coordinates to (x, y) spatial positions.


In [ ]:
rda.rio.transform()

### Coordinate Reference System (CRS)
We `rio.crs` we can get the CRS of our raster and reproject our raster.

In [ ]:
rda.rio.crs

### Reprojection
We can also reproject our raster from one CRS to another.

Lets reproject our raster from "EPSG:6326" to "EPSG:32612"

In [ ]:
rda_reproj = rda.rio.reproject("EPSG:32612")

:::{note}
We have to update our CRS system to the new projection with `rio.write_crs`. We set `inplace=True` to write the CRS to the existing dataset.

In [ ]:
rda_reproj.rio.write_crs("EPSG:32612", inplace=True)

## Exercise

::::{admonition} Exercise
:class: tip

Can you reproject and update the CRS of second band of data to 'ESPG:3857' and plot your results?

:::{admonition} Solution
:class: dropdown

```python
rda[1].rio.reproject("EPSG:3857").rio.write_crs("EPSG:3857", inplace=True).plot()
```
:::
::::
